In [3]:
import subprocess
import os

# Caminho do arquivo SQL dump
dump_file = '/home/Projetos/ProjetosDocker/data/Dump_202625021614.sql'

# Verificar se arquivo existe
if os.path.exists(dump_file):
    file_size = os.path.getsize(dump_file)
    print(f'✓ Arquivo encontrado: {dump_file}')
    print(f'  Tamanho: {file_size / (1024*1024):.2f} MB')
else:
    print(f'✗ Arquivo não encontrado: {dump_file}')

✓ Arquivo encontrado: /home/Projetos/ProjetosDocker/data/Dump_202625021614.sql
  Tamanho: 0.08 MB


In [6]:
# Analisar o dump usando Docker com PostgreSQL 17
result = subprocess.run(
    ['docker', 'run', '--rm', '-i', 'postgres:17', 'pg_restore', '-l'],
    input=open(dump_file, 'rb').read(),
    capture_output=True
)

if result.returncode == 0:
    lines = result.stdout.decode('utf-8').split('\n')
    
    # Extrair metadados do cabeçalho
    print('=== INFORMAÇÕES DO DUMP ===')
    for line in lines[:20]:
        if line.startswith(';'):
            print(line)
    
    # Contar tabelas e objetos
    print('\n=== OBJETOS NO DUMP ===')
    tables = [l for l in lines if 'TABLE' in l and 'public' in l]
    indexes = [l for l in lines if 'INDEX' in l]
    constraints = [l for l in lines if 'CONSTRAINT' in l]
    fks = [l for l in lines if 'FK CONSTRAINT' in l]
    
    print(f'Tabelas: {len([l for l in tables if "TABLE DATA" not in l])}')
    print(f'Índices: {len(indexes)}')
    print(f'Constraints: {len(constraints)}')
    print(f'Foreign Keys: {len(fks)}')
    
    # Listar tabelas
    print('\n=== TABELAS ENCONTRADAS ===')
    table_names = set()
    for line in lines:
        if 'TABLE public ' in line and 'TABLE DATA' not in line:
            parts = line.split()
            if len(parts) >= 4:
                table_names.add(parts[-2])
    
    for table in sorted(table_names):
        print(f'  - {table}')
else:
    print(f'Erro ao ler dump: {result.stderr.decode("utf-8")}')

=== INFORMAÇÕES DO DUMP ===
;
; Archive created at 2026-02-25 16:15:18 UTC
;     dbname: rm_automation
;     TOC Entries: 41
;     Compression: gzip
;     Dump Version: 1.16-0
;     Format: CUSTOM
;     Integer: 4 bytes
;     Offset: 8 bytes
;     Dumped from database version: 18.1
;     Dumped by pg_dump version: 18.1
;
;
; Selected TOC Entries:
;

=== OBJETOS NO DUMP ===
Tabelas: 5
Índices: 15
Constraints: 12
Foreign Keys: 7

=== TABELAS ENCONTRADAS ===
  - conciliacoes_entradas
  - conciliacoes_execucoes
  - extrato_openfinance
  - extrato_openfinance_movimentacoes
  - extrato_openfinance_protocolos


In [7]:
# Extrair informações detalhadas das tabelas (colunas e tipos)
print('\n=== DETALHES DAS TABELAS ===\n')

# Re-executar para obter as linhas novamente
result = subprocess.run(
    ['docker', 'run', '--rm', '-i', 'postgres:17', 'pg_restore', '-l'],
    input=open(dump_file, 'rb').read(),
    capture_output=True
)

lines = result.stdout.decode('utf-8').split('\n')

# Encontrar as seções de CREATE TABLE
current_table = None
for line in lines:
    if 'TABLE public' in line and 'TABLE DATA' not in line:
        parts = line.split()
        if len(parts) >= 4:
            current_table = parts[-2]
            # Buscar índices desta tabela
            table_indexes = [l for l in lines if f'{current_table}_' in l and 'INDEX' in l]
            table_fks = [l for l in lines if current_table in l and 'FK CONSTRAINT' in l]
            
            print(f'📋 {current_table}')
            print(f'   └─ Índices: {len(table_indexes)}')
            if table_fks:
                print(f'   └─ Foreign Keys: {len(table_fks)}')
            print()


=== DETALHES DAS TABELAS ===

📋 conciliacoes_entradas
   └─ Índices: 3
   └─ Foreign Keys: 1

📋 conciliacoes_execucoes
   └─ Índices: 2
   └─ Foreign Keys: 1

📋 extrato_openfinance
   └─ Índices: 10
   └─ Foreign Keys: 5

📋 extrato_openfinance_movimentacoes
   └─ Índices: 3
   └─ Foreign Keys: 2

📋 extrato_openfinance_protocolos
   └─ Índices: 4
   └─ Foreign Keys: 1

